In [1]:
# Load the HR Attrition dataset into a pandas DataFrame for analysis
import pandas as pd

In [5]:
df = pd.read_csv("HR-Employee-Attrition.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [7]:
#Understand dataset size, column types, and check for missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [9]:
# Purpose: Check class distribution of target variable (Attrition)
df['Attrition'].value_counts()

Attrition
No     1233
Yes     237
Name: count, dtype: int64

In [11]:
# Convert target variable (Attrition) from text to numeric format
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

# Check conversion result
df['Attrition'].value_counts()


Attrition
0    1233
1     237
Name: count, dtype: int64

In [13]:
# Separate dataset into input features (X) and target variable (y)
# X = all independent variables used for prediction
# y = Attrition (what we want to predict)

X = df.drop('Attrition', axis=1)
y = df['Attrition']

# Check shapes to confirm split is correct
print("X shape:", X.shape)
print("y shape:", y.shape)


X shape: (1470, 34)
y shape: (1470,)


In [15]:
# Convert categorical (text) columns into numeric form

# Create a copy of X to avoid modifying original data
X_encoded = pd.get_dummies(X, drop_first=True)

# Check new shape after encoding
print("New X shape:", X_encoded.shape)


New X shape: (1470, 47)


In [19]:
# Split data into training and testing sets
# Training set = used to train the model
# Test set = used to evaluate model performance on unseen data

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.2,   # 20% data for testing
    random_state=42  # ensures reproducible results
)

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


X_train: (1176, 47)
X_test: (294, 47)
y_train: (1176,)
y_test: (294,)


In [23]:
# Scale features so all variables have equal importance
# This improves Logistic Regression performance and helps convergence

from sklearn.preprocessing import StandardScaler

# Create scaler
scaler = StandardScaler()

# Fit only on training data, then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaling completed")


Data scaling completed


In [25]:
# Purpose: Train Logistic Regression model on scaled data for better performance

from sklearn.linear_model import LogisticRegression

# Create model
model = LogisticRegression(max_iter=2000)

# Train model using scaled training data
model.fit(X_train_scaled, y_train)

print("Model training completed successfully")


Model training completed successfully


In [29]:
# Purpose: Evaluate how well the model performs on unseen test data

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Make predictions on test set
y_pred = model.predict(X_test_scaled)

# Accuracy score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Confusion matrix (shows correct vs incorrect predictions)
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Detailed performance report
print("\nClassification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.5408163265306123

Confusion Matrix:
 [[128 127]
 [  8  31]]

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.50      0.65       255
           1       0.20      0.79      0.31        39

    accuracy                           0.54       294
   macro avg       0.57      0.65      0.48       294
weighted avg       0.84      0.54      0.61       294



D:\r\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [31]:
# Try a stronger model (Random Forest) to improve prediction performance

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(random_state=42)

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

from sklearn.metrics import accuracy_score

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Accuracy: 0.8775510204081632


In [33]:
# Identify the most important factors driving employee attrition

import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Importance': rf_model.feature_importances_
})

# Sort by importance
feature_importance = feature_importance.sort_values(by='Importance', ascending=False)

# Show top 10 most important features
feature_importance.head(10)


,Feature,Importance
11,MonthlyIncome,0.071784
46,OverTime_Yes,0.061896
1,DailyRate,0.054603
5,EmployeeNumber,0.049249
0,Age,0.048570
19,TotalWorkingYears,0.047738
12,MonthlyRate,0.047392
7,HourlyRate,0.041425
2,DistanceFromHome,0.040048
22,YearsAtCompany,0.036038


In [37]:
# Purpose: Add model predictions to dataset for Power BI visualization

df['Attrition_Predicted'] = rf_model.predict(X_encoded)

# Export dataset for Power BI
df.to_csv("hr_attrition_dashboard.csv", index=False)


In [42]:
# Purpose: Predict attrition for a single employee using correct feature format

# Take one row but KEEP column names
new_employee = X_encoded.iloc[[0]]

# Predict
prediction = rf_model.predict(new_employee)

print("Predicted Attrition:", prediction[0])


Predicted Attrition: 1
